In [0]:
%sql
-- ============================================================
-- DEV CATALOG: CATALOG & SCHEMA COMMENTS
-- ============================================================

COMMENT ON CATALOG dev IS 'Development catalog for the airline booking analytics pipeline. Mirrors the production medallion architecture (landing -> bronze -> silver -> gold) plus an ops layer for data quality and reconciliation reporting.';

COMMENT ON SCHEMA dev.landing IS 'Landing schema: Staging area for raw source files (CSV) as they arrive from upstream feeds before bronze ingestion. No transformations applied; files are picked up here by the bronze Auto Loader pipeline.';

COMMENT ON SCHEMA dev.bronze IS 'Bronze layer: Raw, as-ingested data loaded from landing files via Auto Loader. All columns arrive as strings with source-file and pipeline-run metadata columns. No cleansing or conforming applied.';

COMMENT ON SCHEMA dev.silver IS 'Silver layer: Cleaned and conformed data with proper data types, primary/foreign key enforcement, and per-table quarantine tables capturing DQ rejects. One clean table and one quarantine table per bronze source.';

COMMENT ON SCHEMA dev.gold IS 'Gold layer: Consumption-ready dimensional model: dim_geography, dim_passenger, dim_airline, dim_route, fact_bookings, and the vw_annual_booking_kpis view for annual KPIs.';

COMMENT ON SCHEMA dev.ops IS 'Ops layer: Operational reporting for data quality, pipeline quality gates, gold reconciliation, and rejected-record counts used to verify bronze-to-silver balance.';

-- ============================================================
-- DEV CATALOG: TABLE COMMENTS (bronze, silver, gold, ops)
-- ============================================================

-- ---- BRONZE (8 tables) ----
COMMENT ON TABLE dev.bronze.region IS 'Bronze layer: Raw region reference data ingested from source files. Columns prefixed r_. All values arrive as strings; date format yyyy-M-d. Includes metadata cols: source_file, _ingested_at, pipeline_run_id, _rescued_data.';

COMMENT ON TABLE dev.bronze.nation IS 'Bronze layer: Raw nation reference data ingested from source files. Columns prefixed n_. All values arrive as strings. Includes metadata cols: source_file, _ingested_at, pipeline_run_id, _rescued_data.';

COMMENT ON TABLE dev.bronze.airline IS 'Bronze layer: Raw airline reference data ingested from source files. Columns prefixed al_. All values arrive as strings. Includes metadata cols: source_file, _ingested_at, pipeline_run_id, _rescued_data.';

COMMENT ON TABLE dev.bronze.route IS 'Bronze layer: Raw route reference data ingested from source files. Columns prefixed ro_. Origin/destination embedded in ro_name (e.g. PEK-ADD); no separate origin/destination key columns. All values arrive as strings. Includes metadata cols: source_file, _ingested_at, pipeline_run_id, _rescued_data.';

COMMENT ON TABLE dev.bronze.airline_route IS 'Bronze layer: Raw airline-route mapping data ingested from source files. Columns prefixed ar_. All values arrive as strings. Includes metadata cols: source_file, _ingested_at, pipeline_run_id, _rescued_data.';

COMMENT ON TABLE dev.bronze.passengers IS 'Bronze layer: Raw passenger data ingested from source files. Columns prefixed pa_. All values arrive as strings. Includes metadata cols: source_file, _ingested_at, pipeline_run_id, _rescued_data.';

COMMENT ON TABLE dev.bronze.booking IS 'Bronze layer: Raw booking header data ingested from source files. Columns prefixed bk_. All values arrive as strings. Includes metadata cols: source_file, _ingested_at, pipeline_run_id, _rescued_data.';

COMMENT ON TABLE dev.bronze.flight_booking IS 'Bronze layer: Raw flight booking line data ingested from source files. Columns prefixed fb_. Grain is one row per booking line. All values arrive as strings. Includes metadata cols: source_file, _ingested_at, pipeline_run_id, _rescued_data.';

-- ---- SILVER (16 tables: 8 clean + 8 quarantine) ----
COMMENT ON TABLE dev.silver.region IS 'Silver layer: Cleaned and conformed region dimension. PK check on region key; null/duplicate rejects quarantined. 5 rows. Sourced from dev.bronze.region.';

COMMENT ON TABLE dev.silver.nation IS 'Silver layer: Cleaned and conformed nation dimension. PK check on nation key; FK to region. 25 rows. Sourced from dev.bronze.nation.';

COMMENT ON TABLE dev.silver.airline IS 'Silver layer: Cleaned and conformed airline dimension. PK check on airline key; FK to nation. 10 rows passed, 2 quarantined. Sourced from dev.bronze.airline.';

COMMENT ON TABLE dev.silver.route IS 'Silver layer: Cleaned and conformed route dimension. PK check on route key. 102 rows passed, 3 quarantined. Sourced from dev.bronze.route.';

COMMENT ON TABLE dev.silver.airline_route IS 'Silver layer: Cleaned and conformed airline-route mapping. PK check on composite key; FKs to airline and route. 396 rows passed, 10 quarantined. Sourced from dev.bronze.airline_route.';

COMMENT ON TABLE dev.silver.passengers IS 'Silver layer: Cleaned and conformed passenger dimension. PK check on passenger key; FK to nation. 1502 rows passed, 18 quarantined. Sourced from dev.bronze.passengers.';

COMMENT ON TABLE dev.silver.booking IS 'Silver layer: Cleaned and conformed booking header. PK check on booking key; FKs to passenger and airline. 8040 rows passed, 45 quarantined. Sourced from dev.bronze.booking.';

COMMENT ON TABLE dev.silver.flight_booking IS 'Silver layer: Cleaned and conformed flight booking line data. PK check on composite key; FKs to booking and route. 24976 rows passed, 189 quarantined. Sourced from dev.bronze.flight_booking.';

COMMENT ON TABLE dev.silver.region_quarantine IS 'Silver layer quarantine: Records rejected from silver.region during DQ checks. Contains rejection_reason with concatenated failing rule codes. 0 rows.';

COMMENT ON TABLE dev.silver.nation_quarantine IS 'Silver layer quarantine: Records rejected from silver.nation during DQ checks. Contains rejection_reason with concatenated failing rule codes. 0 rows.';

COMMENT ON TABLE dev.silver.airline_quarantine IS 'Silver layer quarantine: Records rejected from silver.airline during DQ checks. 2 rows rejected for PK/FK violations.';

COMMENT ON TABLE dev.silver.route_quarantine IS 'Silver layer quarantine: Records rejected from silver.route during DQ checks. 3 rows rejected for PK/FK violations.';

COMMENT ON TABLE dev.silver.airline_route_quarantine IS 'Silver layer quarantine: Records rejected from silver.airline_route during DQ checks. 10 rows rejected for PK/FK violations.';

COMMENT ON TABLE dev.silver.passengers_quarantine IS 'Silver layer quarantine: Records rejected from silver.passengers during DQ checks. 18 rows rejected for PK/FK violations.';

COMMENT ON TABLE dev.silver.booking_quarantine IS 'Silver layer quarantine: Records rejected from silver.booking during DQ checks. 45 rows rejected for PK/FK violations.';

COMMENT ON TABLE dev.silver.flight_booking_quarantine IS 'Silver layer quarantine: Records rejected from silver.flight_booking during DQ checks. 189 rows rejected for PK/FK violations.';

-- ---- GOLD (6 objects: 5 tables + 1 view) ----
COMMENT ON TABLE dev.gold.dim_geography IS 'Gold layer: Geography dimension combining nation and region (25 rows). Natural key: nation_key. Includes home_nation and home_region attributes. Sourced from dev.silver.nation and dev.silver.region.';

COMMENT ON TABLE dev.gold.dim_passenger IS 'Gold layer: Passenger dimension (1502 rows). Includes passenger geography (home_nation, home_region). Sourced from dev.silver.passengers joined to dev.silver.nation and dev.silver.region.';

COMMENT ON TABLE dev.gold.dim_airline IS 'Gold layer: Airline dimension (10 rows). Includes airline geography (airline_nation, airline_region) and airline_rating. Sourced from dev.silver.airline joined to dev.silver.nation and dev.silver.region.';

COMMENT ON TABLE dev.gold.dim_route IS 'Gold layer: Route dimension (102 rows). Sourced from dev.silver.route.';

COMMENT ON TABLE dev.gold.fact_bookings IS 'Gold layer: Fact table at booking-line grain (24976 rows). Natural key: booking_id + line_number. FKs: passenger_key, route_key, airline_key. Measures: seats, ticket_price, discount_rate, tax_rate, net_revenue (calculated as ticket_price * (1 - discount_rate), before tax, no seats multiplication). Sourced from dev.silver tables.';

COMMENT ON TABLE dev.gold.vw_annual_booking_kpis IS 'Gold layer view: Annual booking KPIs (1994-1998). Columns: booking_year, annual_net_revenue, distinct_bookings, active_passengers, avg_booking_value. Built on dev.gold.fact_bookings. Net revenue ~19-20M/year.';

-- ---- OPS (4 tables) ----
COMMENT ON TABLE dev.ops.dq_report IS 'Ops layer: Silver-layer data quality report (34 checks). Includes PK null/duplicate checks, FK referential integrity, numeric/date window validations, and NOT_APPLICABLE entries for route origin/destination keys. Sourced from Task 3 silver DQ checks.';

COMMENT ON TABLE dev.ops.dq_pipeline_report IS 'Ops layer: Full pipeline quality gate report (47 checks across 6 categories: bronze ingestion, silver PKs, fact surrogate keys, numeric rules, gold-silver reconciliation at 0.01% tolerance, rejected-record + bronze=silver+quarantine balance). Pipeline-stopping CRITICAL failures listed. All checks PASS on current dev data.';

COMMENT ON TABLE dev.ops.gold_reconciliation IS 'Ops layer: Gold reconciliation report (7 checks). Validates fact row count, net revenue total vs independent silver recompute, and dimension/fact key uniqueness. All checks PASS.';

COMMENT ON TABLE dev.ops.rejected_record_counts IS 'Ops layer: Rejected record counts by silver table. Shows bronze-to-silver reconciliation: bronze = silver + quarantine for each entity. Used by the Task 5 quality gate for balance verification.';